# Hugging Face Kiro Power — Fine-tuning and Deploying Models on AWS Neuron

This notebook tests the SageMaker AI + Neuron MCP server included in the Hugging Face Kiro Power.

**Prerequisites:**
- SageMaker AI Studio JupyterLab Space (ml.c5.xlarge or larger)
- IAM role with SageMaker AI, S3, and ECR access
- HF Hub token (for model metadata)

**Tools tested:**
1. `recommend_instance` — Suggest Neuron instance (fetches from HF Hub)
2. `list_endpoints` — List active SageMaker endpoints
3. `describe_endpoint` — Get endpoint details
4. `delete_endpoint` — Delete an endpoint
5. `describe_training_job` — Get training job status
6. `deploy_model` — Deploy a model (creates real resources)
7. `create_training_job` — Launch a training job (creates real resources)

## 1. Setup

In [ ]:
!sudo apt-get update && sudo apt-get upgrade -y

In [ ]:
# Clone the repo and install
!git clone https://github.com/aws-samples/sample-huggingface-sagemaker-neuron-kiro-power.git
!cd sample-huggingface-sagemaker-neuron-kiro-power/mcp/sagemaker-neuron-server && pip install -e . && pip install peft


In [ ]:
# Install the MCP server package (if not done above)
# !cd sample-huggingface-sagemaker-neuron-kiro-power/mcp/sagemaker-neuron-server && pip install -e . && pip install peft


**Restart the kernel after install** (Kernel -> Restart Kernel), then continue from Step 2.

## 2. Configuration

In [ ]:
import os
os.environ["HF_TOKEN"] = input("Paste your HF token: ")
print("Token set")

In [ ]:
import os
import sagemaker

# AWS
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
os.environ["SAGEMAKER_ROLE_ARN"] = sagemaker.get_execution_role()
os.environ["S3_OUTPUT_PATH"] = f"s3://sagemaker-{os.environ['AWS_DEFAULT_REGION']}-{sagemaker.Session().account_id()}/huggingface-neuron/output"

# Model
os.environ["HF_MODEL_ID"] = "Qwen/Qwen3-0.6B"

# Inference
os.environ["ENDPOINT_NAME"] = "qwen3-finetuned-kiro"
os.environ["INSTANCE_TYPE"] = "ml.inf2.8xlarge"

# Training
os.environ["TRAINING_JOB_NAME"] = "qwen3-0-6b-finetune"
os.environ["TRAINING_INSTANCE_TYPE"] = "ml.trn1.2xlarge"

print("Configuration:")
print(f"  Region:    {os.environ['AWS_DEFAULT_REGION']}")
print(f"  Role:      {os.environ['SAGEMAKER_ROLE_ARN'][:60]}...")
print(f"  Model:     {os.environ['HF_MODEL_ID']}")
print(f"  Endpoint:  {os.environ['ENDPOINT_NAME']}")
print(f"  Instance:  {os.environ['INSTANCE_TYPE']}")
print(f"  Training:  {os.environ['TRAINING_INSTANCE_TYPE']}")
print(f"  Job Name:  {os.environ['TRAINING_JOB_NAME']}")
print(f"  S3 Output: {os.environ['S3_OUTPUT_PATH']}")


## 3. Verify Installation

In [ ]:
from sagemaker_neuron_server import mcp

print("Registered MCP tools:")
for tool in mcp._tool_manager._tools:
    print(f"  - {tool}")

## 4. Test Instance Recommendations

Fetches model metadata from HF Hub API dynamically -- no hardcoded model list.

In [ ]:
from sagemaker_neuron_server.tools.recommend import register_recommend_tools
from sagemaker_neuron_server.tools.endpoint import register_endpoint_tools
from sagemaker_neuron_server.tools.deploy import register_deploy_tools
from sagemaker_neuron_server.tools.training import register_training_tools
from mcp.server.fastmcp import FastMCP
import json

test_mcp = FastMCP("test")
register_recommend_tools(test_mcp)
register_endpoint_tools(test_mcp)
register_deploy_tools(test_mcp)
register_training_tools(test_mcp)
tools = test_mcp._tool_manager._tools

In [ ]:
# Model from env var -- fetches params from HF Hub
result = json.loads(tools["recommend_instance"].fn(model_id=os.environ["HF_MODEL_ID"], use_case="inference"))
print(json.dumps(result, indent=2))
assert result.get("params_source") == "huggingface_hub", "Should fetch from HF Hub"

In [ ]:
# Any HF model -- no hardcoded list needed
result = json.loads(tools["recommend_instance"].fn(model_id="mistralai/Mistral-7B-v0.1", use_case="inference"))
print(json.dumps(result, indent=2))
assert result.get("params_source") == "huggingface_hub", "Should fetch from HF Hub"

In [ ]:
# User-provided params override
result = json.loads(tools["recommend_instance"].fn(model_id="my-private/model", use_case="training", params_billions=13))
print(json.dumps(result, indent=2))
assert result.get("params_source") == "user_provided", "Should use user-provided value"

In [ ]:
# Non-existent model -- should return graceful error
result = json.loads(tools["recommend_instance"].fn(model_id="nonexistent/fake-model-xyz", use_case="inference"))
print(json.dumps(result, indent=2))
assert "error" in result, "Should return error for unknown model"

## 5. Test Endpoint and Job Management

In [ ]:
# List endpoints
result = json.loads(tools["list_endpoints"].fn())
print(json.dumps(result, indent=2))

In [ ]:
# Describe non-existent endpoint -- graceful error handling
result = json.loads(tools["describe_endpoint"].fn(endpoint_name="fake-endpoint-12345"))
print(json.dumps(result, indent=2))
assert "error" in result

In [ ]:
# Delete non-existent endpoint -- graceful error handling
result = json.loads(tools["delete_endpoint"].fn(endpoint_name="fake-endpoint-12345"))
print(json.dumps(result, indent=2))
assert "error" in result

In [ ]:
# Describe training job — graceful error handling
result = json.loads(tools["describe_training_job"].fn(job_name="nonexistent-job-12345"))
print(json.dumps(result, indent=2))
assert "error" in result

## 6. Test Validation (no AWS resources created)

In [ ]:
# Deploy without instance_type -- should return validation error
old_val = os.environ.pop("INSTANCE_TYPE", None)
result = json.loads(tools["deploy_model"].fn(model_id="test/model", endpoint_name="test"))
print("deploy_model missing instance_type:", result)
assert "error" in result
if old_val:
    os.environ["INSTANCE_TYPE"] = old_val

In [ ]:
# Training without s3_output_path -- should return validation error
old_val = os.environ.pop("S3_OUTPUT_PATH", None)
result = json.loads(tools["create_training_job"].fn(model_id="test/model", job_name="test"))
print("create_training_job missing s3_output_path:", result)
assert "error" in result
if old_val:
    os.environ["S3_OUTPUT_PATH"] = old_val

## 7. Fine-tune model on Trainium (creates real AWS resources)

**Requires quota:** ml.trn1.2xlarge for SageMaker AI training

Fine-tunes with LoRA using the optimum-neuron SFT trainer on the wikitext-2 dataset.


In [ ]:
# Fine-tune model on Trainium

result = json.loads(tools["create_training_job"].fn(
    model_id=os.environ["HF_MODEL_ID"],
    job_name=os.environ["TRAINING_JOB_NAME"],
    instance_type=os.environ["TRAINING_INSTANCE_TYPE"],
    role_arn=os.environ["SAGEMAKER_ROLE_ARN"],
    s3_output_path=os.environ["S3_OUTPUT_PATH"],
    hyperparameters=json.dumps({
        "num_train_epochs": "1",
        "per_device_train_batch_size": "1",
        "learning_rate": "2e-5",
    }),
))
print(json.dumps(result, indent=2))

In [ ]:
# Check training job status
job_name = input("Paste training job name (from output above): ")
result = json.loads(tools["describe_training_job"].fn(job_name=job_name))
print(json.dumps(result, indent=2))

In [ ]:
# Clear S3 Neuron compilation cache (optional)

import boto3

s3 = boto3.client("s3")
bucket = f"sagemaker-{os.environ['AWS_DEFAULT_REGION']}-{boto3.client('sts').get_caller_identity()['Account']}"
prefix = "huggingface-neuron/output/neuron-cache/"
paginator = s3.get_paginator("list_objects_v2")
count = 0
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket=bucket, Key=obj["Key"])
        count += 1
print(f"Deleted {count} objects from s3://{bucket}/{prefix}")

In [ ]:
# Verify training job completed and model artifacts exist

import boto3

sm = boto3.client("sagemaker", region_name=os.environ["AWS_DEFAULT_REGION"])

# Find the latest successful job
jobs = sm.list_training_jobs(
    NameContains=os.environ["TRAINING_JOB_NAME"],
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=10
)

for j in jobs["TrainingJobSummaries"]:
    status = j["TrainingJobStatus"]
    print(f"{j['TrainingJobName']}  {status}  {j['CreationTime']}")
    if status == "Completed":
        job = sm.describe_training_job(TrainingJobName=j["TrainingJobName"])
        print(f"\n✅ Successful job: {j['TrainingJobName']}")
        print(f"   Model artifacts: {job['ModelArtifacts']['S3ModelArtifacts']}")
        print(f"   Duration: {(job['TrainingEndTime'] - job['TrainingStartTime']).total_seconds():.0f}s")
        for m in job.get("FinalMetricDataList", []):
            print(f"   {m['MetricName']}: {m['Value']}")
        break


In [ ]:
# Verify LoRA adapter files in model.tar.gz

import tarfile, io, boto3

s3_uri = input("Paste S3 model artifacts URI (from training output): ")
# Parse s3://bucket/key
bucket = s3_uri.split("/")[2]
key = "/".join(s3_uri.split("/")[3:])

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=bucket, Key=key)
with tarfile.open(fileobj=io.BytesIO(obj["Body"].read())) as tar:
    for m in tar.getmembers():
        print(f"{m.size:>10,}  {m.name}")


## 8. Verify DLC Images & Deploy (creates real AWS resources)

**Requires quota:** ml.inf2.8xlarge for SageMaker AI inference

Verifies latest patched container image from ECR before deployment.


In [ ]:
# Verify latest patched DLC images before deployment
import boto3

ecr = boto3.client("ecr", region_name=os.environ["AWS_DEFAULT_REGION"])
repo = "huggingface-pytorch-inference-neuronx"
resp = ecr.describe_images(
    registryId="763104351884",
    repositoryName=repo,
    filter={"tagStatus": "TAGGED"},
    maxResults=50,
)
tags = sorted(
    [(img["imageTags"], img["imagePushedAt"]) for img in resp["imageDetails"] if "imageTags" in img],
    key=lambda x: x[1], reverse=True
)
print(f"{repo}:")
for t, d in tags[:10]:
    print(f"  {d.strftime('%Y-%m-%d')}  {', '.join(t)}")


In [ ]:
# Fetch latest patched image URI for deployment

def get_latest_neuron_image(repo_name, region=None, registry="763104351884"):
    region = region or os.environ["AWS_DEFAULT_REGION"]
    ecr = boto3.client("ecr", region_name=region)
    resp = ecr.describe_images(
        registryId=registry,
        repositoryName=repo_name,
        filter={"tagStatus": "TAGGED"},
        maxResults=50,
    )
    latest = max(
        [img for img in resp["imageDetails"] if "imageTags" in img],
        key=lambda x: x["imagePushedAt"]
    )
    tag = [t for t in latest["imageTags"] if t != "latest"][0] if len(latest["imageTags"]) > 1 else latest["imageTags"][0]
    return f"{registry}.dkr.ecr.{region}.amazonaws.com/{repo_name}:{tag}"

inference_image = get_latest_neuron_image("huggingface-pytorch-inference-neuronx")
print(f"Deploy with: {inference_image}")


In [ ]:
## Prepare fine-tuned model artifacts for Inferentia deployment

import os
bucket = f"sagemaker-{os.environ['AWS_DEFAULT_REGION']}-{os.popen('aws sts get-caller-identity --query Account --output text').read().strip()}"
s3_key = "huggingface-neuron/merged-models/Qwen3-0.6B-finetuned/model.tar.gz"

!mkdir -p /tmp/repack && rm -f /tmp/repack/*
!aws s3 cp s3://{bucket}/{s3_key} /tmp/repack/model.tar.gz
!cd /tmp/repack && tar xzf model.tar.gz && rm model.tar.gz

# Write inference.py
!mkdir -p /tmp/repack/code
with open("/tmp/repack/code/inference.py", "w") as f:
    f.write('''import os
os.chdir("/tmp")
''')

!cd /tmp/repack && tar czf model.tar.gz --exclude='model.tar.gz' *
!aws s3 cp /tmp/repack/model.tar.gz s3://{bucket}/{s3_key}
print("✅ Done")


In [ ]:
# Deploy fine-tuned model on Inferentia2

import sagemaker
from sagemaker.huggingface import HuggingFaceModel

model_data_uri = input("Paste S3 URI of repacked model tar.gz: ")

model = HuggingFaceModel(
    role=sagemaker.get_execution_role(),
    image_uri=inference_image,
    model_data=model_data_uri,
    env={
        "HF_OPTIMUM_SEQUENCE_LENGTH": "4096",
        "HF_OPTIMUM_BATCH_SIZE": "1",
        "HF_AUTO_CAST_TYPE": "bf16",
        "HF_NUM_CORES": "2",
        "NEURON_COMPILE_CACHE_URL": "/tmp/neuron-cache",
        "NEURONX_CACHE": "on",
        "NEURONX_DUMP_TO": "/tmp",
    },
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=os.environ["INSTANCE_TYPE"],
    endpoint_name=os.environ["ENDPOINT_NAME"],
    container_startup_health_check_timeout=900,
    model_data_download_timeout=900,
)
print(f"Endpoint: {predictor.endpoint_name}")

In [ ]:
# Poll endpoint status (run until InService)

import json
result = json.loads(tools["describe_endpoint"].fn(endpoint_name=os.environ["ENDPOINT_NAME"]))
print(json.dumps(result, indent=2))



In [ ]:
# Invoke endpoint (auto-retry during Neuron compilation)

import boto3, json, time, re
from botocore.config import Config

runtime = boto3.client("sagemaker-runtime", region_name=os.environ["AWS_DEFAULT_REGION"], config=Config(read_timeout=600))
prompt = "What is the capital of France?"
formatted = f"<|im_start|>user\n/no_think\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
payload = json.dumps({"inputs": formatted, "parameters": {"max_new_tokens": 512}})

for attempt in range(1, 8):
    try:
        print(f"Attempt {attempt}...")
        response = runtime.invoke_endpoint(
            EndpointName=os.environ["ENDPOINT_NAME"],
            ContentType="application/json",
            Body=payload
        )
        body = json.loads(response["Body"].read())
        text = body[0]["generated_text"] if isinstance(body, list) else body.get("generated_text", str(body))
        # Strip prompt echo, think tags, special tokens
        if text.startswith(formatted):
            text = text[len(formatted):]
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        for tok in ["<|im_end|>", "<|endoftext|>", "<|im_start|>"]:
            text = text.split(tok)[0]
        print(f"Response: {text.strip()}")
        break
    except Exception as e:
        print(f"  Timed out, retrying in 3 min...")
        time.sleep(180)


## 9. Debug & Cleanup

In [ ]:
# Debug: Check endpoint CloudWatch logs (run if deployment fails)

import boto3

endpoint = os.environ["ENDPOINT_NAME"]
logs = boto3.client("logs", region_name=os.environ["AWS_DEFAULT_REGION"])
streams = logs.describe_log_streams(
    logGroupName=f"/aws/sagemaker/Endpoints/{endpoint}",
    orderBy="LastEventTime", descending=True, limit=1
)
events = logs.get_log_events(
    logGroupName=f"/aws/sagemaker/Endpoints/{endpoint}",
    logStreamName=streams["logStreams"][0]["logStreamName"],
    limit=30
)
for e in events["events"]:
    print(e["message"].strip())



In [ ]:
# Cleanup: Delete endpoint (uncomment when done testing)

# import json
# result = json.loads(tools["delete_endpoint"].fn(endpoint_name=os.environ["ENDPOINT_NAME"]))
# print(result)
